# Candidate Fire Spread Prediction Viewer

Reproduce the H4 candidate-mask experiment and inspect next-day spread predictions spatially. The notebook reports both candidate-conditional metrics and full next-day growth metrics that count growth outside radius-5 support as false negatives.

**Important:** test cases are for diagnosis only. Do not choose features or thresholds from these visualizations.

In [ ]:
# Configuration
from pathlib import Path

REPO_ROOT = Path('/home/jlc3q/New_project/TS-Agentic-AI')
CANDIDATE_ROOT = Path('/home/jlc3q/data/SatFire/event_candidates')
SPLIT = 'test'
HISTORY_DAYS = 4
ALLOW_PARTIAL_HISTORY = True
GOES_VARIANT = 'goes_frp_motion_recent_firepred'
TRAIN_SAMPLE = 1_000_000
THRESHOLD = 0.8
RANDOM_SEED = 42

# Available: motion, recent, recent_fuel_weather, recent_full
MODELS_TO_COMPARE = ['motion', 'recent_fuel_weather', 'recent_full']
SELECTED_MODEL = 'recent_fuel_weather'
REFERENCE_MODEL = 'motion'

# mixed, best, worst, high_growth, random, or manual
CASE_SELECTION = 'mixed'
N_CASES = 6
MIN_FULL_GROWTH_PIXELS = 20
FIRE_ID = None
DATE = None

# Focus each figure on the current component with the most supported true growth.
COMPONENT_VIEW = True
ZOOM_TO_ACTIVITY = True
ZOOM_PADDING = 12
SAVE_FIGURES = False
FIGURE_DIR = CANDIDATE_ROOT / 'prediction_viewer_figures'

In [ ]:
import gc
import sys

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from scipy import ndimage
from sklearn.metrics import average_precision_score, roc_auc_score

sys.path.insert(0, str(REPO_ROOT / 'legacy'))
sys.path.insert(0, str(REPO_ROOT / 'legacy' / 'scripts'))

from eval_pred_event_candidate_masks import (
    FIREPRED_CAT,
    FIREPRED_FUEL_WEATHER,
    FIREPRED_NUM,
    GEOMETRY_CAT,
    GEOMETRY_NUM,
    GOES_FRP,
    GOES_RECENT_MOTION,
    GOES_SUBDAILY_MOTION,
    FullGrowthTruth,
    build_model,
    candidate_path,
    date_mask_metrics,
    firewise_metrics,
    goes_history_features,
    history_num_features,
    load_split,
    summarize,
    summarize_firewise,
)

plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 10})
np.random.seed(RANDOM_SEED)

## Experiment plan

- motion: H4 VIIRS history + daily GOES + early/late GOES motion.
- recent_fuel_weather: adds final 3/6-hour GOES motion and fuel/weather variables.
- recent_full: adds all available FirePred variables.
- Fit exactly the classifier and one-million-row training sample used by the evaluation script.
- Reconstruct 256x256 masks with the fixed 0.8 threshold.

In [ ]:
geom_num = GEOMETRY_NUM + history_num_features(HISTORY_DAYS, ALLOW_PARTIAL_HISTORY)
goes_num = GOES_FRP + goes_history_features(HISTORY_DAYS, ALLOW_PARTIAL_HISTORY)
motion_num = geom_num + goes_num + GOES_SUBDAILY_MOTION
recent_num = motion_num + GOES_RECENT_MOTION

MODEL_SPECS = {
    'motion': (motion_num, GEOMETRY_CAT),
    'recent': (recent_num, GEOMETRY_CAT),
    'recent_fuel_weather': (recent_num + FIREPRED_FUEL_WEATHER, GEOMETRY_CAT),
    'recent_full': (recent_num + FIREPRED_NUM, GEOMETRY_CAT + FIREPRED_CAT),
}

unknown = sorted(set(MODELS_TO_COMPARE) - set(MODEL_SPECS))
if unknown:
    raise ValueError(f'Unknown model names: {unknown}')
if SELECTED_MODEL not in MODELS_TO_COMPARE:
    raise ValueError('SELECTED_MODEL must be included in MODELS_TO_COMPARE')

train_path = candidate_path(
    CANDIDATE_ROOT, 'train', 8, 5.0, 1, HISTORY_DAYS,
    ALLOW_PARTIAL_HISTORY, GOES_VARIANT,
)
test_path = candidate_path(
    CANDIDATE_ROOT, SPLIT, 8, 5.0, 1, HISTORY_DAYS,
    ALLOW_PARTIAL_HISTORY, GOES_VARIANT,
)
print('train:', train_path)
print('test :', test_path)
assert train_path.exists(), train_path
assert test_path.exists(), test_path

## Fit models and predict

This is the expensive cell. It reads the large candidate CSV once per selected model, then retains only probabilities and mask coordinates. Use MODELS_TO_COMPARE = ['recent_fuel_weather'] for the fastest run.

In [ ]:
probabilities = {}
global_rows = []
case_df = None

for model_name in MODELS_TO_COMPARE:
    num_features, cat_features = MODEL_SPECS[model_name]
    features = num_features + cat_features
    print()
    print(f'Fitting {model_name}: {len(num_features)} numeric + {len(cat_features)} categorical')

    X_train, y_train, _ = load_split(train_path, features, sample=TRAIN_SAMPLE)
    X_test, y_test, df_test = load_split(test_path, features, include_keys=True)
    model = build_model(num_features, cat_features)
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1].astype(np.float32)

    minimal = df_test[[
        'fire_id', 'date', 'component_id', 'candidate_row',
        'candidate_col', 'label_ignited_next_day',
    ]].reset_index(drop=True)
    if case_df is None:
        case_df = minimal
    else:
        keys = ['fire_id', 'date', 'component_id', 'candidate_row', 'candidate_col']
        if not case_df[keys].equals(minimal[keys]):
            raise RuntimeError('Candidate row order changed between model loads')

    probabilities[model_name] = prob
    global_rows.append({
        'model': model_name,
        'pr_auc': average_precision_score(y_test, prob),
        'roc_auc': roc_auc_score(y_test, prob),
    })
    del X_train, y_train, X_test, y_test, df_test, minimal, model
    gc.collect()

case_df = case_df.reset_index(drop=True)
pd.DataFrame(global_rows)

In [ ]:
# Confirm both candidate-conditional and full-growth metrics.
truth_provider = FullGrowthTruth(SPLIT)
date_metric_frames = []
summary_rows = []

for model_name, prob in probabilities.items():
    date_metrics = date_mask_metrics(
        case_df, prob, 'threshold', THRESHOLD, truth_provider
    )
    date_metrics['model'] = model_name
    date_metric_frames.append(date_metrics)
    fire_metrics = firewise_metrics(
        date_metrics, model=model_name, split=SPLIT,
        method='threshold', value=THRESHOLD,
    )
    summary_rows.append({
        'model': model_name,
        **summarize(date_metrics),
        **summarize_firewise(fire_metrics),
    })

all_date_metrics = pd.concat(date_metric_frames, ignore_index=True)
model_summary = pd.DataFrame(summary_rows).sort_values(
    'fire_full_mean_iou', ascending=False
)
model_summary[[
    'model',
    'fire_mean_iou', 'fire_mean_f1',
    'fire_full_mean_iou', 'fire_full_mean_f1',
    'fire_full_micro_iou', 'fire_full_micro_f1',
    'fire_candidate_coverage_mean', 'fire_candidate_coverage_micro',
]]

## Choose cases

Selection uses full-growth IoU and excludes trivial dates with fewer than MIN_FULL_GROWTH_PIXELS. mixed includes good, bad, improved, regressed, and high-growth cases, then fills to exactly N_CASES after deduplication.

In [ ]:
selected_metrics = all_date_metrics[
    all_date_metrics['model'].eq(SELECTED_MODEL)
].copy()
reference_metrics = all_date_metrics[
    all_date_metrics['model'].eq(REFERENCE_MODEL)
][['fire_id', 'date', 'full_iou']].rename(columns={'full_iou': 'reference_full_iou'})
selected_metrics = selected_metrics.merge(
    reference_metrics, on=['fire_id', 'date'], how='left'
)
selected_metrics['delta_full_iou'] = (
    selected_metrics['full_iou'] - selected_metrics['reference_full_iou']
)
eligible = selected_metrics[
    selected_metrics['full_true_pixels'] >= MIN_FULL_GROWTH_PIXELS
].copy()

if CASE_SELECTION == 'manual':
    if FIRE_ID is None or DATE is None:
        raise ValueError('Set FIRE_ID and DATE for manual selection')
    chosen = selected_metrics[
        selected_metrics['fire_id'].astype(str).eq(str(FIRE_ID))
        & selected_metrics['date'].astype(str).eq(str(DATE))
    ]
elif CASE_SELECTION == 'best':
    chosen = eligible.nlargest(N_CASES, ['full_iou', 'full_true_pixels'])
elif CASE_SELECTION == 'worst':
    chosen = eligible.nsmallest(N_CASES, ['full_iou', 'full_true_pixels'])
elif CASE_SELECTION == 'high_growth':
    chosen = eligible.nlargest(N_CASES, 'full_true_pixels')
elif CASE_SELECTION == 'random':
    chosen = eligible.sample(min(N_CASES, len(eligible)), random_state=RANDOM_SEED)
elif CASE_SELECTION == 'mixed':
    seeds = pd.concat([
        eligible.nlargest(1, ['full_iou', 'full_true_pixels']),
        eligible.nsmallest(1, ['full_iou', 'full_true_pixels']),
        eligible.nlargest(1, 'delta_full_iou'),
        eligible.nsmallest(1, 'delta_full_iou'),
        eligible.nlargest(1, 'full_true_pixels'),
    ]).drop_duplicates(['fire_id', 'date'])
    used = set(zip(seeds['fire_id'].astype(str), seeds['date'].astype(str)))
    remaining = eligible[
        ~eligible.apply(
            lambda row: (str(row['fire_id']), str(row['date'])) in used,
            axis=1,
        )
    ].sort_values(['full_true_pixels', 'full_iou'], ascending=False)
    chosen = pd.concat([seeds, remaining]).drop_duplicates(
        ['fire_id', 'date']
    ).head(N_CASES)
else:
    raise ValueError(f'Unknown CASE_SELECTION: {CASE_SELECTION}')

if chosen.empty:
    raise ValueError('No cases matched the selection')

case_keys = list(chosen[['fire_id', 'date']].itertuples(index=False, name=None))
comparison = all_date_metrics.merge(
    chosen[['fire_id', 'date']].drop_duplicates(), on=['fire_id', 'date'], how='inner'
)
comparison[[
    'model', 'fire_id', 'date',
    'true_pixels', 'iou', 'f1',
    'full_true_pixels', 'candidate_coverage', 'full_iou', 'full_f1',
]].sort_values(['fire_id', 'date', 'model'])

In [ ]:
def selected_component_id(fire_id: str, date: str) -> int:
    rows = case_df[
        case_df['fire_id'].astype(str).eq(str(fire_id))
        & case_df['date'].astype(str).eq(str(date))
    ].copy()
    positions = rows.index.to_numpy(dtype=np.int64)
    rows['selected_score'] = probabilities[SELECTED_MODEL][positions]
    stats = rows.groupby('component_id').agg(
        true_pixels=('label_ignited_next_day', 'sum'),
        max_score=('selected_score', 'max'),
        candidate_pixels=('candidate_row', 'size'),
    )
    return int(stats.sort_values(
        ['true_pixels', 'max_score', 'candidate_pixels'], ascending=False
    ).index[0])

def case_arrays(fire_id: str, date: str, model_name: str, component_id: int | None):
    rows = case_df[
        case_df['fire_id'].astype(str).eq(str(fire_id))
        & case_df['date'].astype(str).eq(str(date))
    ]
    if component_id is not None:
        rows = rows[rows['component_id'].eq(component_id)]
    positions = rows.index.to_numpy(dtype=np.int64)
    rr = rows['candidate_row'].to_numpy(dtype=np.int64)
    cc = rows['candidate_col'].to_numpy(dtype=np.int64)
    labels = rows['label_ignited_next_day'].to_numpy(dtype=bool)

    candidate_support = np.zeros((256, 256), dtype=bool)
    candidate_support[rr, cc] = True
    candidate_true = np.zeros((256, 256), dtype=bool)
    candidate_true[rr[labels], cc[labels]] = True
    score = np.zeros((256, 256), dtype=np.float32)
    np.maximum.at(score, (rr, cc), probabilities[model_name][positions])
    pred_mask = candidate_support & (score >= THRESHOLD)

    current, full_growth = truth_provider.current_and_growth(str(fire_id), str(date))
    if component_id is None:
        current_view = current
        growth_view = full_growth
    else:
        structure = ndimage.generate_binary_structure(2, 2)
        current_labels, _ = ndimage.label(current, structure=structure)
        current_view = current_labels == component_id
        _, nearest = ndimage.distance_transform_edt(~current, return_indices=True)
        nearest_component = current_labels[nearest[0], nearest[1]]
        growth_view = full_growth & (nearest_component == component_id)

    expected_candidate_true = growth_view & candidate_support
    mismatch = int(np.logical_xor(candidate_true, expected_candidate_true).sum())
    if mismatch:
        raise ValueError(
            f'{fire_id} {date} component={component_id}: '
            f'{mismatch} candidate labels disagree with raw growth'
        )
    return (
        current_view, growth_view, candidate_support,
        candidate_true, score, pred_mask,
    )

def activity_bounds(mask: np.ndarray, padding: int):
    coords = np.argwhere(mask)
    if not len(coords):
        return 0, 255, 0, 255
    r0, c0 = coords.min(axis=0)
    r1, c1 = coords.max(axis=0)
    return (
        max(0, int(r0) - padding), min(255, int(r1) + padding),
        max(0, int(c0) - padding), min(255, int(c1) + padding),
    )

context_cmap = ListedColormap(['white', '#4d5156', '#e53935', '#7e57c2'])
error_cmap = ListedColormap([
    'white', '#4d5156', '#2e7d32', '#fb8c00', '#d81b60', '#7e57c2'
])

def plot_case(fire_id: str, date: str):
    component_id = selected_component_id(fire_id, date) if COMPONENT_VIEW else None
    n_models = len(MODELS_TO_COMPARE)
    fig, axes = plt.subplots(
        n_models, 3, figsize=(15, 4.4 * n_models), squeeze=False
    )

    for row_idx, model_name in enumerate(MODELS_TO_COMPARE):
        current, full_growth, candidates, candidate_true, score, pred = case_arrays(
            fire_id, date, model_name, component_id
        )
        supported_growth = full_growth & candidates
        unsupported_growth = full_growth & ~candidates
        context = np.zeros((256, 256), dtype=np.uint8)
        context[current] = 1
        context[supported_growth] = 2
        context[unsupported_growth] = 3

        error = np.zeros((256, 256), dtype=np.uint8)
        error[current] = 1
        error[pred & full_growth] = 2
        error[pred & ~full_growth] = 3
        error[~pred & supported_growth] = 4
        error[unsupported_growth] = 5

        candidate_tp = int((pred & candidate_true).sum())
        candidate_fp = int((pred & ~candidate_true).sum())
        candidate_fn = int((~pred & candidate_true).sum())
        candidate_union = candidate_tp + candidate_fp + candidate_fn
        candidate_iou = candidate_tp / candidate_union if candidate_union else 1.0

        full_tp = int((pred & full_growth).sum())
        full_fp = int((pred & ~full_growth).sum())
        full_fn = int((~pred & full_growth).sum())
        full_union = full_tp + full_fp + full_fn
        full_iou = full_tp / full_union if full_union else 1.0
        full_f1_denom = 2 * full_tp + full_fp + full_fn
        full_f1 = 2 * full_tp / full_f1_denom if full_f1_denom else 1.0
        coverage = supported_growth.sum() / full_growth.sum() if full_growth.any() else 1.0

        axes[row_idx, 0].imshow(
            context, cmap=context_cmap, vmin=0, vmax=3, interpolation='nearest'
        )
        probability_map = np.where(candidates, score, np.nan)
        im = axes[row_idx, 1].imshow(
            probability_map, cmap='inferno', vmin=0, vmax=1, interpolation='nearest'
        )
        if current.any():
            axes[row_idx, 1].contour(
                current, levels=[0.5], colors='cyan', linewidths=0.6
            )
        if full_growth.any():
            axes[row_idx, 1].contour(
                full_growth, levels=[0.5], colors='lime', linewidths=0.8
            )
        fig.colorbar(im, ax=axes[row_idx, 1], fraction=0.046, pad=0.04)

        axes[row_idx, 2].imshow(
            error, cmap=error_cmap, vmin=0, vmax=5, interpolation='nearest'
        )
        axes[row_idx, 2].set_title(
            f'full IoU={full_iou:.3f} F1={full_f1:.3f} | '
            f'coverage={coverage:.3f} | candidate IoU={candidate_iou:.3f}'
        )
        axes[row_idx, 0].set_ylabel(model_name, fontsize=10)

        if row_idx == 0:
            axes[row_idx, 0].set_title('VIIRS context')
            axes[row_idx, 1].set_title('Candidate probability')
        if ZOOM_TO_ACTIVITY:
            bounds_mask = current | full_growth | candidates
            r0, r1, c0, c1 = activity_bounds(bounds_mask, ZOOM_PADDING)
            for ax in axes[row_idx]:
                ax.set_xlim(c0, c1)
                ax.set_ylim(r1, r0)
        for ax in axes[row_idx]:
            ax.set_xticks([])
            ax.set_yticks([])

    legend = [
        Patch(color='#4d5156', label='current component'),
        Patch(color='#2e7d32', label='TP'),
        Patch(color='#fb8c00', label='FP'),
        Patch(color='#d81b60', label='supported FN'),
        Patch(color='#7e57c2', label='unsupported FN'),
    ]
    fig.legend(handles=legend, loc='lower center', ncol=5, frameon=False)
    component_text = f'component={component_id}' if component_id is not None else 'all components'
    fig.suptitle(
        f'{fire_id} | {date} -> next VIIRS day | {component_text} | threshold={THRESHOLD}',
        y=0.995,
    )
    fig.tight_layout(rect=(0, 0.04, 1, 0.98))
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(
            FIGURE_DIR / f'{fire_id}_{date}_{component_text}_prediction.png',
            dpi=180, bbox_inches='tight',
        )
    plt.show()

## Prediction maps

The error panel now treats purple unsupported growth as false negatives. Full IoU/F1 therefore evaluates the complete component growth, while candidate IoU remains available only as a diagnostic conditional metric.

In [ ]:
for fire_id, date in case_keys:
    plot_case(str(fire_id), str(date))

## Interpretation checklist

- Compare full IoU/F1, not candidate IoU, against image models.
- Candidate coverage is the maximum recall/IoU ceiling imposed by radius-5 support.
- Purple unsupported FN indicates that candidate generation, not the classifier, must be expanded.
- Compare components separately when a fire ID contains distant simultaneous starts.
- Use validation and paired fire-wise tests for decisions; use test plots only for failure analysis.